# Robust iSWAP with modulated coupling (non-local frame)

## Background: two equivalent formulations

Starting from the Duffing oscillator Hamiltonian and performing the rotating wave approximation (RWA), the **non-local frame** (Case 2) gives:

$$H(t) = \frac{\delta_{12}}{2} Z_2 + g(t)\big[(X_1X_2+Y_1Y_2)\cos(\delta_{12}t) + (Y_1X_2-X_1Y_2)\sin(\delta_{12}t)\big]$$
$$\quad + u_{x1}X_1 + u_{y1}Y_1 + u_{x2}X_2 + u_{y2}Y_2$$

with 5 physical controls $\{g(t), u_{x1}, u_{y1}, u_{x2}, u_{y2}\}$.

The coupling term $g(t) M(t)$ has an **explicitly time-modulated operator**:
$$M(t) = (X_1X_2+Y_1Y_2)\cos(\delta_{12}t) + (Y_1X_2-X_1Y_2)\sin(\delta_{12}t)$$

---

### Formulation 1 – Time-dependent H (Part 1 below)

Use one coupling control $g(t)$ with the full time-dependent Hamiltonian.
Piccolo's `TimeDependentBilinearIntegrator` handles this correctly in the NLP.
→ Works for **fidelity-only** optimization.

### Formulation 2 – Quadrature decomposition (Part 2 below)

Substitute $v_I(t) = g(t)\cos(\delta_{12}t)$ and $v_Q(t) = g(t)\sin(\delta_{12}t)$.
Then the coupling becomes a **static bilinear form**:
$$v_I(t)(X_1X_2+Y_1Y_2) + v_Q(t)(Y_1X_2-X_1Y_2)$$

**The modulation is baked into the control parameterization**: the optimizer freely chooses $v_I$ and $v_Q$ at each knot point. If it wants to realise a physical modulated coupling, it can set $v_I(k) = g_k\cos(\delta_{12}t_k)$ and $v_Q(k) = g_k\sin(\delta_{12}t_k)$. Without this constraint, the optimizer has strictly more freedom — it can also access the extra degrees of freedom opened up by making $v_I$ and $v_Q$ independent.

Critically, this works with QuantumCollocation's `UnitaryVariationalProblem`, enabling **fidelity + robustness** optimization.

## Setup

In [ ]:
import Pkg; Pkg.activate(".."); Pkg.instantiate();
Pkg.develop(path="../../../QuantumCollocation.jl")
using PiccoloQuantumObjects
using QuantumCollocation
using LinearAlgebra
using SparseArrays
using Statistics
using CairoMakie
using Random
using NamedTrajectories
⊗ = kron

In [ ]:
# --- Physical parameters ---
# δ12: detuning between qubits in the non-local frame (rad/ns, matching Δt units)
# Set δ12 = 0 to recover the local (resonant) frame Case 1 as a sanity check.
δ12 = 2π * 0.05   # 50 MHz detuning (adjust to your device)

# --- Two-qubit Pauli operators ---
XI = GATES.X ⊗ GATES.I
YI = GATES.Y ⊗ GATES.I
ZI = GATES.Z ⊗ GATES.I
IX = GATES.I ⊗ GATES.X
IY = GATES.I ⊗ GATES.Y
IZ = GATES.I ⊗ GATES.Z
XX = GATES.X ⊗ GATES.X
YY = GATES.Y ⊗ GATES.Y
XY = GATES.X ⊗ GATES.Y
YX = GATES.Y ⊗ GATES.X
ZZ = GATES.Z ⊗ GATES.Z

# --- iSWAP target gate ---
U_goal = exp(1.0im * π/4 * (XX + YY))

# --- Dephasing error channels (robust against these) ---
∂ₑH = [ZI, IZ, ZZ];

## Part 1 – Fidelity optimization with time-dependent modulated coupling

Uses one coupling control $g(t)$ with the full non-local frame Hamiltonian.
This uses Piccolo's `TimeDependentBilinearIntegrator` for correctness in the NLP.

In [ ]:
# Non-local frame: one coupling control g(t) modulating both XX+YY and YX-XY.
# Controls: [g, uX1, uY1, uX2, uY2]
function H_nonlocal(u, t)
    g, uX1, uY1, uX2, uY2 = u
    return (
        (δ12/2) * IZ
        + g  * ((XX + YY) * cos(δ12 * t) + (YX - XY) * sin(δ12 * t))
        + uX1 * XI  + uY1 * YI
        + uX2 * IX  + uY2 * IY
    )
end

T    = 50
Δt   = 0.2      # ns
T_f  = T * Δt   # total gate time (ns)
a_bound = 5.0
bounds_mod = fill(a_bound, 5)  # same bound for all 5 controls

# QuantumSystem from functional Hamiltonian (T_f required by QuantumCollocation API)
sys_td = QuantumSystem(H_nonlocal, T_f, fill(a_bound, 5))

In [ ]:
# Fidelity-only optimisation (default smooth pulse problem, no robustness)
Random.seed!(42)
prob_td = UnitarySmoothPulseProblem(
    sys_td, U_goal, T, Δt;
    Δt_max  = Δt, Δt_min = Δt,
    a_bound = a_bound,
    Q_t     = 1.0
)
push!(prob_td.constraints,
    FinalUnitaryFidelityConstraint(U_goal, :Ũ⃗, 0.9999, prob_td.trajectory)
)
solve!(prob_td; max_iter=500, options=IpoptOptions(eval_hessian=false))
println("Part 1 fidelity: ", unitary_rollout_fidelity(prob_td.trajectory, sys_td))

## Part 2 – Robust optimization via quadrature decomposition

**Key idea**: rewrite the modulated coupling using two independent controls:
$$v_I(t)(X_1X_2+Y_1Y_2) + v_Q(t)(Y_1X_2-X_1Y_2)$$
where $v_I = g\cos(\delta_{12}t)$, $v_Q = g\sin(\delta_{12}t)$.

The Hamiltonian is now a **static sum of drives** — compatible with
`UnitaryVariationalProblem` and its adjoint-based robustness objective.

Controls: $\{u_{x1}, u_{y1}, u_{x2}, u_{y2}, v_I, v_Q\}$ (6 total).

The drift $\frac{\delta_{12}}{2}Z_2$ is included exactly.

In [ ]:
# --- Quadrature-decomposed drives ---
# Coupling quadratures (the two Duffing coupling channels)
M_I = XX + YY   # in-phase coupling  (X₁X₂ + Y₁Y₂)
M_Q = YX - XY   # quadrature coupling (Y₁X₂ - X₁Y₂)

# Drift: qubit detuning from non-local frame
H_drift = (δ12 / 2) * IZ

# Six drives: 2 microwave per qubit + 2 coupling quadratures
H_drive = [XI, YI, IX, IY, M_I, M_Q]

# Build the system for fidelity rollout and as base for variational
sys_quad = QuantumSystem(H_drift, H_drive)

# Build variational system: same dynamics + error channels
varsys = VariationalQuantumSystem(H_drift, H_drive, ∂ₑH)

var_count = length(∂ₑH)  # 3 error channels: ZI, IZ, ZZ
println("n_drives = ", sys_quad.n_drives, "  n_errors = ", var_count)

In [ ]:
# --- Variational (adjoint) robust problem ---
# Minimises EV = ||U† ∂U/∂ε||² for each error channel,
# subject to a fidelity constraint F ≥ 0.9999.

F_min    = 0.9999
ä_bound  = 1.0    # acceleration bound (sweep this for Pareto frontier)
num_iter = 1000

Random.seed!(42)
var_prob = UnitaryVariationalProblem(
    varsys, U_goal, T, Δt;
    robust_times  = [[T] for _ in 1:var_count],  # penalise at final knot
    Δt_max        = Δt, Δt_min = Δt,
    a_bound       = a_bound,
    dda_bound     = ä_bound,
    Q             = 0.0,   # no infidelity penalty; use constraint instead
    Q_r           = 1.0,   # robustness weight
    Q_s           = 0.0,
    piccolo_options = PiccoloOptions(verbose=false)
)
push!(var_prob.constraints,
    FinalUnitaryFidelityConstraint(U_goal, :Ũ⃗, F_min, var_prob.trajectory)
)
solve!(var_prob; max_iter=num_iter, options=IpoptOptions(eval_hessian=false))
println("Part 2 fidelity: ", unitary_rollout_fidelity(var_prob.trajectory, sys_quad))

## Evaluate and compare susceptibilities

Compute $\mathcal{E}_V(E)$ for every two-qubit Pauli string.

In [ ]:
# var_obj: adjoint susceptibility metric  |Tr[(U†∂U)†(U†∂U)]| / ((T-1)Δt)² / d
function var_obj(
    traj::NamedTrajectory,
    H_drives::Vector{Matrix{ComplexF64}},
    H_drift::Matrix{ComplexF64},
    H_errors::Vector{Matrix{ComplexF64}}
)
    Δt_val = traj.Δt[1]
    T_val  = traj.T
    vs     = VariationalQuantumSystem(H_drift, H_drives, H_errors)
    Ũ⃗, ∂Ũ⃗ = variational_unitary_rollout(traj, vs)
    U  = iso_vec_to_operator(Ũ⃗[:, end])
    ∂U = iso_vec_to_operator(∂Ũ⃗[1][:, end])
    d  = size(U, 1)
    return abs(tr((U'∂U)'*(U'∂U))) / ((T_val - 1) * Δt_val)^2 / d
end

# All 15 two-qubit Pauli strings
pauli_labels = ["XI","YI","ZI","IX","IY","IZ","XX","XY","XZ","YX","YY","YZ","ZX","ZY","ZZ"]
pauli_ops    = [XI, YI, ZI, IX, IY, IZ, XX, XY, GATES.X⊗GATES.Z,
                YX, YY, GATES.Y⊗GATES.Z, GATES.Z⊗GATES.X, GATES.Z⊗GATES.Y, ZZ]

function eval_susceptibilities(traj, drives, drift, ops)
    return [var_obj(traj, drives, drift, [op]) for op in ops]
end

susc_var = eval_susceptibilities(var_prob.trajectory, H_drive, H_drift, pauli_ops)

println("\nSusceptibility for targeted errors (ZI, IZ, ZZ):")
for (lbl, val) in zip(["ZI","IZ","ZZ"], [susc_var[3], susc_var[6], susc_var[15]])
    @printf("  ε(%s) = %.3e\n", lbl, val)
end

## Pareto frontier: robustness vs acceleration bound

Sweep $\ddot{u}$ bound to trace the robustness–controllability trade-off,
analogous to Fig. 6 in the paper.

In [ ]:
ä_vals = exp10.(range(-2, stop=3, length=10))
var_avg_vec = Float64[]
var_F_vec   = Float64[]

for ä in ä_vals
    Random.seed!(42)
    prob = UnitaryVariationalProblem(
        varsys, U_goal, T, Δt;
        robust_times    = [[T] for _ in 1:var_count],
        Δt_max = Δt, Δt_min = Δt,
        a_bound         = a_bound,
        dda_bound       = ä,
        Q               = 0.0, Q_r = 1.0, Q_s = 0.0,
        piccolo_options = PiccoloOptions(verbose=false)
    )
    push!(prob.constraints,
        FinalUnitaryFidelityConstraint(U_goal, :Ũ⃗, F_min, prob.trajectory)
    )
    solve!(prob; max_iter=1000, options=IpoptOptions(eval_hessian=false))

    F = unitary_rollout_fidelity(prob.trajectory, sys_quad)
    push!(var_F_vec, F)

    susc = eval_susceptibilities(prob.trajectory, H_drive, H_drift, [ZI, IZ, ZZ])
    push!(var_avg_vec, mean(susc))

    @printf("ä=%.2e  F=%.6f  avg_susc=%.3e\n", ä, F, var_avg_vec[end])
end

## Plots

In [ ]:
# --- Plot 1: optimised pulse (Part 2 robust solution) ---
fig1, axes = plot(var_prob.trajectory; merge_labels=true)
fig1

In [ ]:
# --- Plot 2: recover the effective g(t) from quadrature controls ---
# The physical coupling amplitude is sqrt(v_I² + v_Q²) at each knot.
traj = var_prob.trajectory
a    = traj[:a]               # (n_drives × T) matrix
vI   = a[5, :]                # v_I control (index 5 = M_I drive)
vQ   = a[6, :]                # v_Q control (index 6 = M_Q drive)
ts   = cumsum([0.0; traj.Δt[1:end-1]])

fig2 = Figure(size=(800, 400))
ax   = Axis(fig2[1,1]; xlabel="t (ns)", ylabel="amplitude",
            title="Recovered coupling controls")
lines!(ax, ts, vI; label="v_I (in-phase, XX+YY)",       color=:blue)
lines!(ax, ts, vQ; label="v_Q (quadrature, YX-XY)",    color=:orange)
lines!(ax, ts, sqrt.(vI.^2 .+ vQ.^2); label="|g(t)| = √(v_I²+v_Q²)",
       color=:green, linestyle=:dash)
axislegend(ax; position=:rt)
fig2

In [ ]:
# --- Plot 3: Pareto frontier ---
fig3 = Figure(size=(600, 450))
ax   = Axis(fig3[1,1];
            xlabel = "ü constraint",
            ylabel = "⅓(ε(ZI)+ε(IZ)+ε(ZZ))",
            xscale = log10, yscale = log10,
            title  = "Modulated drive: robustness vs curvature bound")
lines!(ax, ä_vals, var_avg_vec; color=:blue, linewidth=2)
scatter!(ax, ä_vals, var_avg_vec; color=:blue, marker=:circle, markersize=8)
fig3